# M04-02 -- KPIS



In [8]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col

spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
sales = (
    fact.join(customers, "customer_id", "inner")
    .where(col("is_billable"))
)
print(sales.count())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
1122


## Paso 2 -- Cuatro métrica globales



In [9]:
from pyspark.sql.functions import sum as fsum, countDistinct, round as fround

kpis = sales.agg(
    fround(fsum("gmv_line"), 2).alias("gmv"),
    countDistinct("order_id").alias("orders"),
)
kpis = kpis.withColumn("aov", fround(col("gmv") / col("orders"), 2))
kpis.show()


+---------+------+------+
|      gmv|orders|   aov|
+---------+------+------+
|400157.73|   469|853.21|
+---------+------+------+



## Paso 3 -- Tasa de cancelación



In [10]:
from pyspark.sql.functions import avg

orders = spark.read.parquet(str(STAGING / "orders_clean"))
print ("orders")
ord_ok = orders.join(customers, "customer_id", "inner")
print ("orders ok")
ord_ok.count()
cancel = ord_ok.agg(
    avg((col("status") == "cancelled").cast("double")).alias("cancel_rate")
)
print ("pedidos cancelados")
cancel.count()
cancel.show()
print("pedidos con cliente", ord_ok.count())

orders
orders ok
pedidos cancelados
+------------------+
|       cancel_rate|
+------------------+
|0.2153846153846154|
+------------------+

pedidos con cliente 780


## Paso 4 Kpi por canal



In [11]:
(
    sales.groupBy("channel_norm")
    .agg(
        fround(fsum("gmv_line"), 2).alias("gmv"),
        countDistinct("order_id").alias("orders"),
    )
    .orderBy(col("gmv").desc())
    .show()
)

+------------+---------+------+
|channel_norm|      gmv|orders|
+------------+---------+------+
|         web|156374.29|   176|
|         app|144693.39|   175|
|       other| 56537.03|    60|
|       store| 42553.03|    58|
+------------+---------+------+



## Mejora - gmv por mes y país



In [13]:
(
    sales.groupBy("order_month", "country")
    .agg(fround(fsum("gmv_line"),2 ).alias("gmv"))
    .orderBy ("order_month", "country")
    .show(10)
)

+-----------+-------+--------+
|order_month|country|     gmv|
+-----------+-------+--------+
|    2024-01|     DE| 3487.51|
|    2024-01|     ES|  5412.7|
|    2024-01|     FR| 5356.23|
|    2024-01|     IT|  7064.3|
|    2024-01|     PT| 2596.75|
|    2024-02|     DE| 4439.18|
|    2024-02|     ES|10103.47|
|    2024-02|     FR| 6428.67|
|    2024-02|     IT| 5426.95|
|    2024-02|     PT| 2500.93|
+-----------+-------+--------+
only showing top 10 rows

